### Скачивание данных

In [2]:
!mkdir -p data/images

# !wget -O /tmp/isic2020_train.zip https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_JPEG.zip
# !unzip -q -j /tmp/isic2020_train.zip "*.jpg" -d data/images
# !rm /tmp/isic2020_train.zip

# !wget -O /tmp/isic2019_train.zip https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_Input.zip
# !unzip -q -j /tmp/isic2019_train.zip "*.jpg" -d data/images
# !rm /tmp/isic2019_train.zip

!wget -O /tmp/isic2019_test.zip https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip
!unzip -q -n -j /tmp/isic2019_test.zip "*.jpg" -d data/images
!rm /tmp/isic2019_test.zip

--2026-06-14 13:53:58--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_Input.zip
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 16.15.212.114, 16.15.223.28, 54.231.204.89, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|16.15.212.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3823247963 (3.6G) [application/zip]
Saving to: ‘/tmp/isic2019_test.zip’

/tmp/isic2019_test. 100%[===================>]   3.56G  46.5MB/s    in 79s     

2026-06-14 13:55:17 (46.4 MB/s) - ‘/tmp/isic2019_test.zip’ saved [3823247963/3823247963]



In [3]:
!wget https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
!wget https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv
!wget https://isic-archive.s3.amazonaws.com/challenges/2020/ISIC_2020_Training_GroundTruth.csv

--2026-06-14 13:56:08--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Training_GroundTruth.csv
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 52.217.201.73, 16.15.183.238, 16.15.183.170, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|52.217.201.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1291479 (1.2M) [text/csv]
Saving to: ‘ISIC_2019_Training_GroundTruth.csv’

ISIC_2019_Training_ 100%[===================>]   1.23M  3.32MB/s    in 0.4s    

2026-06-14 13:56:09 (3.32 MB/s) - ‘ISIC_2019_Training_GroundTruth.csv’ saved [1291479/1291479]

--2026-06-14 13:56:09--  https://isic-archive.s3.amazonaws.com/challenges/2019/ISIC_2019_Test_GroundTruth.csv
Resolving isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)... 16.15.214.112, 16.182.33.81, 52.217.68.100, ...
Connecting to isic-archive.s3.amazonaws.com (isic-archive.s3.amazonaws.com)|16.15.214.112|:443... connected.
HTTP

In [476]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import tqdm
import os
from IPython.display import clear_output
from typing import Optional, Dict, List

import torch.utils.data as data
import torchvision.transforms.v2 as v2
import torch
import torch.nn as nn
from PIL import Image

import wandb
import json


sns.set_theme(context="notebook", palette="muted")
RANDOM_STATE = 33

In [5]:
target2019 = pd.read_csv('ISIC_2019_Training_GroundTruth.csv')
target2019test = pd.read_csv('ISIC_2019_Test_GroundTruth.csv')
target2020 = pd.read_csv('ISIC_2020_Training_GroundTruth.csv')

In [6]:
target2020 = target2020[['image_name', 'target']].rename(columns={'image_name' : 'image'})
target2020['target'] = target2020['target'].astype('float64')
target2019 = target2019[['MEL', 'image']].rename(columns={'MEL' : 'target'})
target2019test = target2019test[['MEL', 'image']].rename(columns={'MEL' : 'target'})

In [7]:
labels = pd.concat([target2020, target2019, target2019test], axis=0)
labels = target2019test

### Класс датасета

In [ ]:
class ISICDataset(data.Dataset):
    def __init__(self, root, labels, transforms):
        super().__init__()
        self.transforms = transforms
        self.files = [root + '/' + fname + '.jpg' for fname in labels['image'].tolist()]
        self.targets = labels['target'].tolist()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        fname = self.files[index]
        with Image.open(fname) as im:
            im = im.convert('RGB')
        
        if self.transforms:
            im = self.transforms(im)
        
        return im, self.targets[index]

### Функции для обучения

In [478]:
def training_epoch(model : nn.Module, optimizer : torch.optim.Optimizer, criterion : nn.Module, train_loader : data.DataLoader, device : torch.device, tqdm_desc='Train'):
    model.train()

    loss_sum = 0.0
    n_samples = 0

    all_probs = []
    all_targets = []

    pbar = tqdm.tqdm(train_loader, desc=tqdm_desc)
    for x, y in pbar:
        x = x.to(device).float()
        y = y.to(device).float()

        optimizer.zero_grad()

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        bs = x.size(0)
        loss_sum += float(loss.item()) * bs
        n_samples += bs

        all_targets.extend(y.cpu().detach().numpy().flatten().tolist())
        all_probs.extend(logits.sigmoid().cpu().detach().numpy().flatten().tolist())

        avg_loss = loss_sum / max(1, n_samples)

        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})

    return {'loss' : loss_sum / max(1, n_samples), 'roc_auc' : roc_auc_score(all_targets, all_probs)}


@torch.no_grad()
def validation_epoch(model : nn.Module, criterion : nn.Module, valid_loader : data.DataLoader, device : torch.device, tqdm_desc='Valid'):
    model.eval()

    loss_sum = 0.0
    n_samples = 0

    all_probs = []
    all_targets = []

    pbar = tqdm.tqdm(valid_loader, desc=tqdm_desc)
    for x, y in pbar:
        x = x.to(device).float()
        y = y.to(device).float()

        logits = model(x).squeeze(1)
        loss = criterion(logits, y)

        bs = x.size(0)
        loss_sum += float(loss.item()) * bs
        n_samples += bs

        all_targets.extend(y.cpu().detach().numpy().flatten().tolist())
        all_probs.extend(logits.sigmoid().cpu().detach().numpy().flatten().tolist())

        avg_loss = loss_sum / max(1, n_samples)

        pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    

    return {'loss' : loss_sum / max(1, n_samples), 'roc_auc' : roc_auc_score(all_targets, all_probs)}

def plot_metrics(train_losses, val_losses, train_auc_roc, val_auc_roc):
    clear_output(True)
    epochs = np.arange(1, len(train_losses) + 1)

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss', marker='o')
    plt.plot(epochs, val_losses, label='Val Loss', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Loss Dynamics')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_auc_roc, label='Train AUC-ROC', marker='o')
    plt.plot(epochs, val_auc_roc, label='Val AUC-ROC', marker='o')
    plt.xlabel('Epoch')
    plt.ylabel('AUC-ROC')
    plt.title('AUC-ROC Dynamics')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.ylim([-0.05, 1.05])
    plt.legend()

    plt.tight_layout()
    plt.show()

def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer, epoch: int, val_loss : float) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'epoch': epoch, 'val_loss': val_loss, 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()}, path)

def fit(model : nn.Module, train_loader : data.DataLoader, val_loader : data.DataLoader, optimizer : torch.optim.Optimizer, criterion : nn.Module, device : torch.device, num_epochs: int, out_dir: str = None, plot_fn=None, use_wandb: bool = False, wandb_project: str = None, wandb_run_name: Optional[str] = None, wandb_config: Optional[dict] = None, save_every_n_epochs : int = 1) -> Dict[str, List[float]]:
    model.to(device)

    history = {
        'train_loss': [],
        'val_loss': [],
        'train_roc_auc' : [],
        'val_roc_auc' : [],
    }

    best_roc_auc = 0

    run = None
    if use_wandb:
        run = wandb.init(project=wandb_project, name=wandb_run_name, config=wandb_config if wandb_config else {})
        wandb.config.update({'num_params': sum(p.numel() for p in model.parameters()), 'device': str(device)}, allow_val_change=True)

    for epoch in range(1, num_epochs + 1):
        tr = training_epoch(model=model, train_loader=train_loader, optimizer=optimizer, criterion=criterion, device=device, tqdm_desc=f'Train {epoch}/{num_epochs}')
        va = validation_epoch(model=model, valid_loader=val_loader, criterion=criterion, device=device, tqdm_desc=f'Valid {epoch}/{num_epochs}',)

        history['train_loss'].append(tr['loss'])
        history['train_roc_auc'].append(tr['roc_auc'])
        history['val_loss'].append(va['loss'])
        history['val_roc_auc'].append(va['roc_auc'])
        
        if epoch % save_every_n_epochs == 0: save_checkpoint(os.path.join(out_dir, 'checkpoints', f"epoch_{epoch:03d}.pt"), model, optimizer, epoch, va['loss'])
        if va['roc_auc'] >= best_roc_auc: save_checkpoint(os.path.join(out_dir, 'checkpoints', 'best.pt'), model, optimizer, epoch, va['loss'])

        print(f'[{epoch:02d}/{num_epochs}] train: loss={tr["loss"]:.4f}, roc_auc={tr["roc_auc"]:.4f} | val: loss={va["loss"]:.4f}, roc_auc={va["roc_auc"]:.4f}')
        
        if plot_fn:
            plot_fn(history['train_loss'], history['val_loss'], history['train_roc_auc'], history['val_roc_auc'])

        if use_wandb:
            wandb.log({'epoch': epoch, 'train/loss': tr['loss'], 'train/roc_auc': tr['roc_auc'], 'val/loss': va['loss'], 'val/roc_auc': va['roc_auc']}, step=epoch)

        os.makedirs(f'{out_dir}/logs/', exist_ok=True)
        with open(f'{out_dir}/logs/history.json', 'w', encoding='utf-8') as f:
            json.dump(history, f, ensure_ascii=False, indent=4)

    os.makedirs(os.path.dirname(f'{out_dir}/data/'), exist_ok=True)
    torch.save(train_loader.dataset.tensors, f'{out_dir}/data/train_dataset.pt')
    torch.save(val_loader.dataset.tensors, f'{out_dir}/data/val_dataset.pt')

    if use_wandb and run is not None:
        run.finish()

    return history

#### разделим выборку на train, val и test

In [468]:
train_labels, test_labels = train_test_split(labels, test_size=0.3, random_state=RANDOM_STATE, stratify=labels['target'])
test_labels, val_labels = train_test_split(test_labels, test_size=0.35, random_state=RANDOM_STATE, stratify=test_labels['target'])

In [469]:
len(train_labels['target']), len(test_labels['target']), len(val_labels['target'])

(5766, 1606, 866)

посчитаем статистики по трейн сету

In [ ]:
def get_stats(loader):
    csum = torch.zeros(3)
    csqsum = torch.zeros(3)
    num_pix = 0

    for b, _ in tqdm.tqdm(loader):
        csum += b.sum(dim=[0, 2, 3])
        csqsum += (b ** 2).sum(dim=[0, 2, 3])
        num_pix += b.shape[0] * b.shape[2] * b.shape[3]
    
    return csum / num_pix, torch.sqrt(csqsum / num_pix - (csum / num_pix) ** 2)

train_mean, train_std = get_stats(data.DataLoader(ISICDataset('data/images', train_labels, transforms=v2.Compose([v2.Resize((256, 256)), v2.ToImage(), v2.ToDtype(torch.float32, True)])), batch_size=1024))

100%|██████████| 6/6 [02:39<00:00, 26.53s/it]


In [471]:
train_mean, train_std

(tensor([0.6047, 0.5060, 0.5026]), tensor([0.2580, 0.2319, 0.2407]))

In [484]:
train_transforms = v2.Compose([
    v2.Resize((256, 256)),
    v2.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), ratio=(0.9, 1.1), interpolation=v2.InterpolationMode.BILINEAR),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(degrees=180, interpolation=v2.InterpolationMode.BILINEAR, fill=0),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.03),
    v2.RandomApply([v2.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))], p=0.2),
    v2.RandomErasing(p=0.3, scale=(0.02, 0.05), ratio=(0.5, 2)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, True),
    v2.Normalize(train_mean, train_std)
])
train_dataset = ISICDataset('data/images', train_labels, transforms=train_transforms)

test_transforms = v2.Compose([v2.Resize((256, 256)), v2.ToImage(), v2.ToDtype(torch.float32, True), v2.Normalize(train_mean, train_std)])

test_dataset = ISICDataset('data/images', test_labels, transforms=test_transforms)
val_dataset = ISICDataset('data/images', val_labels, transforms=test_transforms)

criterion = nn.BCEWithLogitsLoss(torch.tensor(len(train_labels['target']) / sum(train_labels['target'])))

train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = data.DataLoader(test_dataset, batch_size=64)
val_loader = data.DataLoader(val_dataset, batch_size=64, num_workers=2)

model = nn.Sequential(
    nn.Conv2d(3, 8, 3, 1, 1),
    nn.GELU(),
    nn.Conv2d(8, 8, 3, 1, 1),
    nn.GELU(),
    nn.Conv2d(8, 8, 3, 1, 1),
    nn.GELU(),
    nn.Conv2d(8, 8, 3, 1, 1),
    nn.GELU(),
    nn.Conv2d(8, 64, 3, 1, 1),
    nn.Flatten(),
    nn.LazyLinear(32),
    nn.GELU(),
    nn.Linear(32, 1)
)

device = torch.device('cpu')

optimizer = torch.optim.Adam(model.parameters())

In [ ]:
fit(model, train_loader, val_loader, optimizer, criterion, device, 1)

Train 1/1:   4%|▍         | 8/181 [01:13<22:59,  7.97s/it, loss=47.7416]

: 